In [0]:
import requests
import pandas as pd
from pyspark.sql import functions as F

base_url = "https://hubeau.eaufrance.fr/api/v1/qualite_eau_potable/resultats_dis"
code_departement = "59"
size = 1000

page = 1
all_data = []

while True:
    url = f"{base_url}?code_departement={code_departement}&page={page}&size={size}"
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    json_data = response.json()
    data = json_data.get("data", [])
    if not data:
        break
    all_data.extend(data)
    print(f"Page: {page} | URL: {url}")
    print(f"Status: {response.status_code}")
    if len(data) < size:
        break
    page += 1
pdf = pd.DataFrame(all_data)
pdf = pdf.where(pd.notnull(pdf), None)
df = spark.createDataFrame(pdf)
df = df.withColumn("source_url", F.lit(base_url)) \
       .withColumn("ingestion_type", F.lit("api")) \
       .withColumn("ingested_at", F.current_timestamp())
hash_cols = [c for c in df.columns if c not in ["ingested_at"]]

df = df.withColumn(
    "hash",
    F.sha2(F.to_json(F.struct(*hash_cols)), 256)
)

df = df.withColumn(
    "annee",
    F.year(F.to_timestamp("date_prelevement"))
)
df.write.format("delta") \
    .mode("append") \
    .partitionBy("annee") \
    .saveAsTable("main.bronze.qualite_eau_region")

display(df)

In [0]:
%pip install great_expectations

In [0]:
%sql
DESCRIBE TABLE main.bronze.qualite_eau_region;

In [0]:
%sql
SELECT * FROM main.bronze.qualite_eau_region LIMIT 10;